# 04 — Rate spread and the validation gap

**Goal:** Both baselines predict about 30% too few defaults on validation (notebook 03). Test whether the interest rate environment at origination explains it, then look for what does. Validation only; no test data is used.

## Setup

The loan table now has `market_rate` (FRED 30-year or 15-year mortgage rate in the origination month) and `rate_spread` (loan rate minus market rate).

In [ ]:
from dataclasses import replace

import duckdb
import pandas as pd

from mortgage_credit_risk.config import load_config
from mortgage_credit_risk.data import sample_paths
from mortgage_credit_risk.evaluate import binary_metrics, c_index
from mortgage_credit_risk.macro import load_series
from mortgage_credit_risk.models import (
    default_by,
    fit_cox,
    fit_logistic,
    hazard_ratios,
    horizon_label,
)
from mortgage_credit_risk.pipeline import build_model_tables

cfg = load_config()
months_path, loans_path = build_model_tables(cfg, cfg.build_rate)
loans = pd.read_parquet(loans_path)
H = cfg.logistic_horizon_months
labeled = horizon_label(loans, H)
train = loans[loans["split"] == "train"]
val = loans[loans["split"] == "validation"]
l_train = labeled[labeled["split"] == "train"]
v12 = labeled[labeled["split"] == "validation"]

## Rates by split

How much of the rise in loan rates was the market, and how much was borrower pricing?

In [ ]:
loans.groupby("split")[["orig_rate", "market_rate", "rate_spread"]].mean().round(3)

## Does the rate spread close the gap?

Three feature sets, everything else the same. Ratio is predicted / observed defaults by age 12.

In [ ]:
variants = {
    "A: orig_rate": ("credit_score", "ltv", "dti", "orig_rate"),
    "B: rate_spread": ("credit_score", "ltv", "dti", "rate_spread"),
    "C: rate_spread + market_rate": (
        "credit_score",
        "ltv",
        "dti",
        "rate_spread",
        "market_rate",
    ),
}
rows, ratios = {}, {}
for name, numeric in variants.items():
    v_cfg = replace(cfg, numeric_features=numeric)
    features = [*numeric, *v_cfg.categorical_features]
    logit, cox = fit_logistic(l_train, v_cfg), fit_cox(train, v_cfg)
    m_logit = binary_metrics(v12["target"], logit.predict_proba(v12[features])[:, 1])
    m_cox = binary_metrics(v12["target"], default_by(cox, v12, H))
    rows[name] = {
        "logistic_auc": m_logit["auc"],
        "logistic_ratio": m_logit["ratio"],
        "cox_auc": m_cox["auc"],
        "cox_c_index": c_index(val, cox.predict(val[features])),
        "cox_ratio": m_cox["ratio"],
    }
    ratios[name] = hazard_ratios(cox)["hazard_ratio"]
pd.DataFrame(rows).T.round(3)

In [ ]:
pd.DataFrame(ratios).round(2)

## The economy after origination

If jobs or house prices got worse in 2017 and 2018, that could explain more defaults. Yearly averages from FRED.

In [ ]:
unemployment = load_series("UNRATE", cfg.fred_dir)
prices = load_series("CSUSHPISA", cfg.fred_dir)
pd.DataFrame(
    {
        "unemployment_pct": unemployment.groupby(unemployment.index.year).mean(),
        "house_price_change_pct": prices.groupby(prices.index.year).mean().pct_change()
        * 100,
    }
).loc[2015:2020].round(1)

## Where the gap is

Predicted vs observed 12-month defaults by state group and DTI band, using the Cox model from notebook 03.

In [ ]:
cox = fit_cox(train, cfg)


def gap_by(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    """Defaults, predicted defaults and their ratio by group."""
    table = frame.groupby(column, observed=True).agg(
        loans=("p", "size"), defaults=("target", "sum"), predicted=("p", "sum")
    )
    table["ratio"] = table["predicted"] / table["defaults"]
    table["share_of_loans_pct"] = 100 * table["loans"] / table["loans"].sum()
    return table.round(2)


tables = {}
for split in ["train", "validation"]:
    frame = labeled[labeled["split"] == split].copy()
    frame["p"] = default_by(cox, frame, H)
    frame["state_group"] = frame["state"].where(
        frame["state"].isin(["FL", "TX"]), "other"
    )
    frame["dti_band"] = pd.cut(frame["dti"], [0, 43, 45, 50])
    tables[split] = frame
    print(split)
    print(gap_by(frame, "state_group"), "\n")

In [ ]:
gap_by(tables["validation"], "dti_band")

## Hurricanes

Harvey hit Texas in Aug 2017 and Irma hit Florida in Sep 2017. How many validation defaults were in disaster relief at some point?

In [ ]:
con = duckdb.connect()
con.sql(f"""
    WITH defaults AS (
        SELECT loan_id, state, event_period FROM '{loans_path}'
        WHERE split = 'validation' AND event = 1
    ),
    flags AS (
        SELECT d.loan_id, any_value(d.state) AS state,
            any_value(d.event_period) AS event_period,
            coalesce(bool_or(m.disaster_flag = 'Y'), false) AS disaster
        FROM '{sample_paths(cfg, cfg.build_rate)[1]}' m
        JOIN defaults d USING (loan_id)
        GROUP BY 1
    )
    SELECT CASE WHEN state IN ('FL', 'TX') THEN state ELSE 'other' END AS state_group,
        count(*) AS n_default,
        count(*) FILTER (WHERE event_period >= 201801) AS in_2018,
        count(*) FILTER (WHERE disaster) AS ever_in_disaster_relief
    FROM flags GROUP BY 1 ORDER BY 1
""").df()

## Findings

Numbers are for the 5% build sample, validation only.

1. **Most of the rise in loan rates was the market, not the borrowers.** Validation loan rates were 0.34 points above training. The market rate explains 0.28 of that. The rate spread (borrower pricing) rose only 0.06.

2. **The rate spread is a real risk signal.** Its Cox hazard ratio is 1.32 per standard deviation, as strong as DTI. Loans priced above the market default more.

3. **The rate spread does not close the gap. It makes it slightly worse.** With the spread instead of the loan rate, the Cox ratio drops from 0.71 to 0.61. The old model got part of the gap right by accident: it read validation's higher market rates as riskier borrowers. Adding the market rate back (variant C) gives about the same results as the loan rate, because loan rate = spread + market rate.

4. **The economy after origination cannot explain it either.** In 2017 and 2018, unemployment fell (4.4% to 3.9%) and house prices rose about 6% a year. Both point to fewer defaults, not more.

5. **Looser lending is not the cause.** Loans with DTI 45 to 50 are predicted well (ratio 0.96). The gap is in ordinary loans.

6. **The gap is almost all Florida and Texas.** Validation ratio is 0.19 in Florida (41 defaults, 8 predicted) and 0.53 in Texas (19 vs 10). Every other state is 1.00. In training, Florida and Texas were predicted well (0.96 and 1.08). Florida and Texas are 14% of validation loans but 41% of its 12-month defaults.

7. **These are hurricane defaults.** Harvey (Texas, Aug 2017) and Irma (Florida, Sep 2017). 87% of Florida's and 65% of Texas's validation defaults were in disaster relief at some point. Rule B ignores late months during relief, but many loans were still 90+ days late after relief ended, and those months count.

8. **No origination feature could predict this.** The hurricanes came after the loans were made. The models are fine for the rest of the country.

**Decision:**

- Keep rule B and the notebook 03 features (`orig_rate`). Report the hurricane effect as a known shock instead of changing the default rule or recalibrating.
- Keep `market_rate` and `rate_spread` in the tables for later use.